# Complete 95% VaR Analysis Pipeline

This notebook downloads adjusted daily prices for SPY, QQQ, GLD, and VOO from `yfinance`, stores VIX separately as a covariate, computes daily returns, tests normality, estimates rolling one-day VaR with three methods, and visualizes violations with VIX below each return chart.

Forecasts are shifted forward one day: the VaR threshold for day `t` uses only the prior 250 trading days.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats as scipy_stats
from scipy.stats import chi2, norm
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.6f}".format)


## 1. Configuration

In [ ]:
TICKERS = ["SPY", "QQQ", "GLD", "VOO"]
VIX_TICKER = "^VIX"

START_DATE = "2015-01-01"
END_DATE = "2025-12-31"
YF_END_DATE = "2026-01-01"  # yfinance end dates are exclusive.

VAR_LEVEL = 0.95
ALPHA = 1 - VAR_LEVEL
WINDOW = 250
N_SIMS = 10_000
SEED = 42

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

for directory in [RAW_DIR, PROCESSED_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

config = pd.Series({
    "tickers": ", ".join(TICKERS),
    "vix_covariate": VIX_TICKER,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "var_level": VAR_LEVEL,
    "alpha": ALPHA,
    "rolling_window_days": WINDOW,
    "monte_carlo_paths_per_day": N_SIMS,
    "random_seed": SEED,
})
display(config.to_frame("value"))


## 2. Download Adjusted Prices and VIX

In [ ]:
def get_adjusted_close(tickers):
    """Download adjusted close prices and return a tidy DataFrame."""
    raw = yf.download(
        tickers,
        start=START_DATE,
        end=YF_END_DATE,
        auto_adjust=False,
        actions=False,
        progress=False,
        group_by="column",
        threads=True,
    )
    if raw.empty:
        raise RuntimeError(f"No data returned for {tickers}.")

    if isinstance(raw.columns, pd.MultiIndex):
        first_level = raw.columns.get_level_values(0)
        if "Adj Close" in first_level:
            adjusted = raw["Adj Close"].copy()
        elif "Close" in first_level:
            adjusted = raw["Close"].copy()
        else:
            raise KeyError("Neither 'Adj Close' nor 'Close' found in yfinance response.")
    else:
        price_col = "Adj Close" if "Adj Close" in raw.columns else "Close"
        ticker_name = tickers if isinstance(tickers, str) else tickers[0]
        adjusted = raw[[price_col]].rename(columns={price_col: ticker_name})

    if isinstance(adjusted, pd.Series):
        ticker_name = tickers if isinstance(tickers, str) else adjusted.name
        adjusted = adjusted.to_frame(ticker_name)

    adjusted = adjusted.sort_index()
    adjusted.index = pd.to_datetime(adjusted.index).tz_localize(None)
    if not isinstance(tickers, str):
        adjusted = adjusted[[ticker for ticker in tickers if ticker in adjusted.columns]]
    return adjusted


price_cache = RAW_DIR / "adjusted_close_prices.csv"
vix_cache = RAW_DIR / "vix_adjusted_close.csv"

try:
    prices = get_adjusted_close(TICKERS)
    vix = get_adjusted_close(VIX_TICKER)
except Exception as exc:
    if price_cache.exists() and vix_cache.exists():
        print(f"Download failed: {exc}. Loading cached CSV files instead.")
        prices = pd.read_csv(price_cache, index_col=0, parse_dates=True)
        vix = pd.read_csv(vix_cache, index_col=0, parse_dates=True)
    else:
        raise

prices = prices.dropna(how="all").ffill().dropna()
vix = vix.dropna(how="all").ffill().dropna()
if vix.shape[1] == 1:
    vix.columns = [VIX_TICKER]

vix = vix.reindex(prices.index).ffill().bfill()

prices.to_csv(price_cache)
vix.to_csv(vix_cache)

print(f"Price sample: {prices.index.min().date()} to {prices.index.max().date()} | {len(prices):,} rows")
display(prices.tail())
display(vix.tail())


## 3. Daily Returns

In [ ]:
returns = prices.pct_change().dropna(how="any")
vix_aligned = vix.reindex(returns.index).ffill().bfill()

returns.to_csv(PROCESSED_DIR / "daily_returns.csv")
vix_aligned.to_csv(PROCESSED_DIR / "vix_aligned.csv")

print(f"Return sample: {returns.index.min().date()} to {returns.index.max().date()} | {len(returns):,} rows")
display(returns.tail())


## 4. Summary Statistics and Normality Tests

In [ ]:
summary_stats = returns.agg(["mean", "std", "skew", "kurt"]).T
summary_stats = summary_stats.rename(columns={"kurt": "excess_kurtosis"})
summary_stats.index.name = "ticker"
summary_stats.to_csv(PROCESSED_DIR / "return_summary_statistics.csv")

display(summary_stats)

normality_rows = []
for ticker in TICKERS:
    x = returns[ticker].dropna()
    jb = scipy_stats.jarque_bera(x)
    dagostino = scipy_stats.normaltest(x)
    normality_rows.append({
        "ticker": ticker,
        "observations": len(x),
        "jarque_bera_stat": jb.statistic,
        "jarque_bera_p_value": jb.pvalue,
        "dagostino_k2_stat": dagostino.statistic,
        "dagostino_k2_p_value": dagostino.pvalue,
        "reject_normality_5pct_jb": jb.pvalue < 0.05,
        "reject_normality_5pct_k2": dagostino.pvalue < 0.05,
    })

normality_tests = pd.DataFrame(normality_rows).set_index("ticker")
normality_tests.to_csv(PROCESSED_DIR / "normality_tests.csv")
display(normality_tests)


## 5. QQ Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
for ticker, ax in zip(TICKERS, axes.flat):
    scipy_stats.probplot(returns[ticker].dropna(), dist="norm", plot=ax)
    ax.set_title(f"{ticker} QQ Plot vs Normal")
    ax.grid(True, alpha=0.35)

fig.suptitle("Daily Return QQ Plots", fontsize=16)
qq_path = FIGURE_DIR / "qq_plots_returns.png"
fig.savefig(qq_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved QQ plots to {qq_path.relative_to(PROJECT_ROOT)}")


## 6. Rolling VaR Estimation

Methods:

- Historical simulation: rolling 5th percentile of realized returns.
- Parametric normal: rolling mean plus rolling standard deviation multiplied by the 5th percentile of the standard normal distribution.
- Monte Carlo: 10,000 normal draws per forecast date using the rolling mean and standard deviation, then the simulated 5th percentile.

In [ ]:
MODEL_COLUMNS = {
    "Historical": "VaR_Historical",
    "Parametric Normal": "VaR_Normal",
    "Monte Carlo": "VaR_MonteCarlo",
}


def monte_carlo_var(rolling_mean, rolling_std, rng, chunk_size=250):
    """Simulate N_SIMS one-day returns per date and take the lower alpha quantile."""
    valid = rolling_mean.notna() & rolling_std.notna()
    valid_index = rolling_mean.index[valid]
    out = pd.Series(index=rolling_mean.index, dtype=float, name="VaR_MonteCarlo")

    for start in range(0, len(valid_index), chunk_size):
        idx = valid_index[start:start + chunk_size]
        mu = rolling_mean.loc[idx].to_numpy()
        sigma = rolling_std.loc[idx].to_numpy()
        simulations = rng.normal(loc=mu[:, None], scale=sigma[:, None], size=(len(idx), N_SIMS))
        out.loc[idx] = np.quantile(simulations, ALPHA, axis=1)
    return out


def compute_var_for_ticker(return_series, rng):
    rolling_mean = return_series.rolling(WINDOW, min_periods=WINDOW).mean().shift(1)
    rolling_std = return_series.rolling(WINDOW, min_periods=WINDOW).std(ddof=1).shift(1)
    var_historical = return_series.rolling(WINDOW, min_periods=WINDOW).quantile(ALPHA).shift(1)
    var_normal = rolling_mean + rolling_std * norm.ppf(ALPHA)
    var_mc = monte_carlo_var(rolling_mean, rolling_std, rng)

    out = pd.DataFrame({
        "return": return_series,
        "rolling_mean": rolling_mean,
        "rolling_std": rolling_std,
        "VaR_Historical": var_historical,
        "VaR_Normal": var_normal,
        "VaR_MonteCarlo": var_mc,
    })
    out.index.name = "Date"
    return out


rng = np.random.default_rng(SEED)
var_by_ticker = {ticker: compute_var_for_ticker(returns[ticker], rng) for ticker in TICKERS}
var_estimates = pd.concat(var_by_ticker, names=["ticker", "Date"])
var_estimates.to_csv(PROCESSED_DIR / "rolling_var_estimates.csv")

display(var_estimates.groupby(level=0).tail(3))


## 7. Violation Backtest Summary

In [ ]:
def kupiec_pof_test(n_violations, n_obs, alpha):
    """Kupiec unconditional coverage likelihood-ratio test."""
    if n_obs == 0:
        return np.nan, np.nan
    violation_rate = np.clip(n_violations / n_obs, 1e-12, 1 - 1e-12)
    log_likelihood_null = (n_obs - n_violations) * np.log(1 - alpha) + n_violations * np.log(alpha)
    log_likelihood_hat = (n_obs - n_violations) * np.log(1 - violation_rate) + n_violations * np.log(violation_rate)
    lr_stat = -2 * (log_likelihood_null - log_likelihood_hat)
    p_value = 1 - chi2.cdf(lr_stat, df=1)
    return lr_stat, p_value


backtest_rows = []
for ticker, df in var_by_ticker.items():
    for model_name, column in MODEL_COLUMNS.items():
        valid = df[column].notna()
        n_obs = int(valid.sum())
        violations = df.loc[valid, "return"] < df.loc[valid, column]
        n_violations = int(violations.sum())
        lr_stat, p_value = kupiec_pof_test(n_violations, n_obs, ALPHA)
        backtest_rows.append({
            "ticker": ticker,
            "model": model_name,
            "observations": n_obs,
            "violations": n_violations,
            "expected_violations": ALPHA * n_obs,
            "violation_rate": n_violations / n_obs if n_obs else np.nan,
            "expected_rate": ALPHA,
            "kupiec_lr_stat": lr_stat,
            "kupiec_p_value": p_value,
        })

backtest_summary = pd.DataFrame(backtest_rows)
backtest_summary.to_csv(PROCESSED_DIR / "var_backtest_summary.csv", index=False)
display(backtest_summary)


## 8. Returns, VaR Lines, Violations, and VIX

In [ ]:
PLOT_COLORS = {
    "Actual return": "#303030",
    "Historical": "#005f73",
    "Parametric Normal": "#9b2226",
    "Monte Carlo": "#ee9b00",
    "VIX": "#4b5563",
    "Violation": "#d62828",
}


def plot_var_with_vix(ticker, df, vix_series):
    model_cols = list(MODEL_COLUMNS.values())
    plot_df = df.loc[df[model_cols].notna().any(axis=1)].copy()
    vix_plot = vix_series.reindex(plot_df.index).ffill().bfill()

    fig, (ax_ret, ax_vix) = plt.subplots(
        2,
        1,
        figsize=(15, 8),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08},
    )

    ax_ret.plot(
        plot_df.index,
        plot_df["return"],
        color=PLOT_COLORS["Actual return"],
        linewidth=0.75,
        alpha=0.75,
        label="Actual return",
    )
    for model_name, column in MODEL_COLUMNS.items():
        ax_ret.plot(
            plot_df.index,
            plot_df[column],
            linewidth=1.3,
            label=f"{model_name} VaR",
            color=PLOT_COLORS[model_name],
        )

    any_var_threshold = plot_df[model_cols].max(axis=1)
    violations = plot_df["return"] < any_var_threshold
    ax_ret.scatter(
        plot_df.index[violations],
        plot_df.loc[violations, "return"],
        color=PLOT_COLORS["Violation"],
        s=18,
        zorder=5,
        label="Violation",
    )

    ax_ret.axhline(0, color="#111111", linewidth=0.7, alpha=0.35)
    ax_ret.set_title(f"{ticker}: Actual Daily Returns vs 95% VaR Forecasts")
    ax_ret.set_ylabel("Daily return")
    ax_ret.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
    ax_ret.legend(ncol=3, frameon=True, fontsize=9, loc="lower left")
    ax_ret.grid(True, alpha=0.25)

    ax_vix.plot(vix_plot.index, vix_plot, color=PLOT_COLORS["VIX"], linewidth=1.0, label="VIX")
    ax_vix.fill_between(vix_plot.index, vix_plot.to_numpy(), color="#94a3b8", alpha=0.25)
    ax_vix.set_ylabel("VIX")
    ax_vix.set_xlabel("Date")
    ax_vix.legend(frameon=True, fontsize=9, loc="upper left")
    ax_vix.grid(True, alpha=0.25)

    fig.tight_layout()
    output_path = FIGURE_DIR / f"{ticker}_var_vix.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()
    return output_path


vix_series = vix_aligned[VIX_TICKER]
figure_paths = {ticker: plot_var_with_vix(ticker, var_by_ticker[ticker], vix_series) for ticker in TICKERS}
display(pd.Series({ticker: str(path.relative_to(PROJECT_ROOT)) for ticker, path in figure_paths.items()}, name="saved_figure"))


## 9. Completion Check

In [ ]:
key_outputs = [
    RAW_DIR / "adjusted_close_prices.csv",
    RAW_DIR / "vix_adjusted_close.csv",
    PROCESSED_DIR / "daily_returns.csv",
    PROCESSED_DIR / "return_summary_statistics.csv",
    PROCESSED_DIR / "normality_tests.csv",
    PROCESSED_DIR / "rolling_var_estimates.csv",
    PROCESSED_DIR / "var_backtest_summary.csv",
    FIGURE_DIR / "qq_plots_returns.png",
    *figure_paths.values(),
]

output_check = pd.DataFrame({
    "path": [str(path.relative_to(PROJECT_ROOT)) for path in key_outputs],
    "exists": [path.exists() for path in key_outputs],
})
display(output_check)

if not output_check["exists"].all():
    missing = output_check.loc[~output_check["exists"], "path"].tolist()
    raise FileNotFoundError(f"Missing expected outputs: {missing}")

print("VaR pipeline complete.")
